In [ ]:
import torch
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import gc
import time

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True


config = {
    'dataset': {
        'image_size': 384,
        'batch_size': 6,
        'workers': 4
    },
    'model': {
        'filters': 48,  
        'skip_connections': True
    },
    'training': {
        'epochs': 25,  
        'lr': 0.0002
    }
}

class NYUDepthDataset(Dataset): #an abstract classin pytorch that cant be initialised/ made object of so need to be inherited to use the methods
    def __init__(self, root_folder, split='train', transform=None): #constructor for the class and contains the data directory , is it train ot test and tranformations 
        self.root_folder = root_folder #storing the data directory, split type and image size  in instance variable 
        self.split = split
        self.transform = transform
        self.img_size = config['dataset']['image_size']
        #forming the path to access the images - rgb and depth maps according to my dataset 
        self.split_folder = os.path.join(root_folder, split) #contains the path for train and test folder 
        if split == 'test':
            self.split_folder = os.path.join(self.split_folder, 'official')#test folder has a additional subfolder official that is why official was added 
        
        self.rgb_files = [] #empty list but will contain the paths of the images and as a list can be acessed by indexing 
        self.depth_files = []
        
        try:
            if split == 'train':#since train folder contains subfolders named scenes 
                scene_folders = [d for d in os.listdir(self.split_folder) 
                             if os.path.isdir(os.path.join(self.split_folder, d))] #will contain the names of the scene in the train folder 
                
                for scene in scene_folders: #access indivsual scene names 
                    scene_folder = os.path.join(self.split_folder, scene) #till split_folder we had the path till the train folder now to go in the scenes , scene_folder will contain path till the scene name so inside it is structured like depth mapsssss and then startsrgb 
                    rgb_files = sorted([f for f in os.listdir(scene_folder) if f.startswith('rgb_')]) # so to differentiate into rgb and depth we check for starting name 
                    depth_files = sorted([f for f in os.listdir(scene_folder) if f.startswith('depth_')])
                    
                   
                    rgb_files = rgb_files[::3] #taking every 3rd image from the dataset the dataset contain around 90000+ images will be too much to train
                    depth_files = depth_files[::3]#contains the name of the rgb and depth images
                    
                    self.rgb_files.extend([os.path.join(scene_folder, f) for f in rgb_files]) #now it contains the whole path to the image and 
                    #will be accessible even outside the loop
                    self.depth_files.extend([os.path.join(scene_folder, f) for f in depth_files])
                    
            else:#if the split value is test then 
                self.rgb_files = sorted([os.path.join(self.split_folder, f) for f in os.listdir(self.split_folder) 
                                        if f.startswith('rgb_')])
                self.depth_files = sorted([os.path.join(self.split_folder, f) for f in os.listdir(self.split_folder) 
                                          if f.startswith('depth_')])
            
            assert len(self.rgb_files) == len(self.depth_files) 
            print("equal image pairs")#just to be sure there are equal airs of images 
            
        except Exception as e:
            print(f"Error initializing dataset: {str(e)}")
            self.rgb_files = []
            self.depth_files = []
        
    def __len__(self):
        return len(self.rgb_files) #number of sample in the list- coumtimg only rgb files as thee is corresponding depth file to each rgb file 

    def __getitem__(self, idx): #the index is automatically handled by dataloader-pytorch increments it
        try:
            rgb_path = self.rgb_files[idx] #is a list containg the paths of the images 
            depth_path = self.depth_files[idx]
                
            #RGB image
            rgb = cv2.imread(rgb_path)
            rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
            
            #depth image
            depth = cv2.imread(depth_path, cv2.IMREAD_GRAYSCALE) #opencv reads the depth image as a 3 channel gray image cintaing all the 3 same channels and to work with tensors we require 1 channel gray images 
            
            # Resize to make it even and all the images the same size , added interpolation - which predits the pixel if it is not present while resizing - used INTER_CUBIC which works on the neighbouring 16 pixels (4 x 4) grid
            rgb = cv2.resize(rgb, (self.img_size, self.img_size), interpolation=cv2.INTER_CUBIC)
            depth = cv2.resize(depth, (self.img_size, self.img_size), interpolation=cv2.INTER_CUBIC)
            
            # tensors- pytorch functions work on tensors and not numpy arrays 
            rgb = torch.from_numpy(rgb).float().permute(2, 0, 1) / 255.0
            depth = torch.from_numpy(depth).float().unsqueeze(0) / 255.0
            
            # data augumentaion to make the data sample more diverse 
            if self.split == 'train' and random.random() > 0.5:
                # horizontal flipping
                rgb = torch.flip(rgb, dims=[2])
                depth = torch.flip(depth, dims=[2]) #dim=2 it is flipped from left to right if had been 1 or -2 then vertically
            
            return rgb, depth
        except Exception as e:
            print(f"Error processing image at index {idx}: {str(e)}")
            # Return a placeholder image instead of crashing
            placeholder_rgb = torch.zeros(3, self.img_size, self.img_size) #an empty image and depth image sent back if the there is problem in access the image at that index - an tensor with 3 channels for rgb and 1 channel for depth filled with zero sent back 
            placeholder_depth = torch.zeros(1, self.img_size, self.img_size)
            return placeholder_rgb, placeholder_depth
            
class ImprovedDepthModel(nn.Module): #base class that will contain the layers or framework for CNN 
    def __init__(self): #cnstructor
        super(ImprovedDepthModel, self).__init__() #inheriting from the module class 
        
        # the first layer of encoder - in this model i took the number of filters as 48 
        self.down1 = nn.Sequential(
            nn.Conv2d(3, config['model']['filters'], kernel_size=7, stride=2, padding=3),#after this the layer that had 3 channels will be 48 feature maps stacked 
            nn.BatchNorm2d(config['model']['filters']),#normalise
            nn.ReLU(inplace=True),#add non-linearity why? no matter how many layers we add it will still be a simgle linear layer
            nn.Conv2d(config['model']['filters'], config['model']['filters'], kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool2d(2) #maxpooling
        
        self.down2 = nn.Sequential(
            nn.Conv2d(config['model']['filters'], config['model']['filters']*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']*2, config['model']['filters']*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*2),
            nn.ReLU(inplace=True)
        )
        self.pool2 = nn.MaxPool2d(2)
        
        self.down3 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*2, config['model']['filters']*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']*4, config['model']['filters']*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*4),
            nn.ReLU(inplace=True)
        )
        self.pool3 = nn.MaxPool2d(2)
        
        self.down4 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*4, config['model']['filters']*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']*8, config['model']['filters']*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*8),
            nn.ReLU(inplace=True)
        )
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(config['model']['filters']*8, config['model']['filters']*16, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*16),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']*16, config['model']['filters']*16, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*16),
            nn.ReLU(inplace=True)
        )
        
        # Decoder part with better upsampling and double convolutions
        self.up4 = nn.ConvTranspose2d(config['model']['filters']*16, config['model']['filters']*8, kernel_size=4, stride=2, padding=1)
        self.conv4 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*16, config['model']['filters']*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']*8, config['model']['filters']*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*8),
            nn.ReLU(inplace=True)
        )
        
        self.up3 = nn.ConvTranspose2d(config['model']['filters']*8, config['model']['filters']*4, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*8, config['model']['filters']*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']*4, config['model']['filters']*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*4),
            nn.ReLU(inplace=True)
        )
        
        self.up2 = nn.ConvTranspose2d(config['model']['filters']*4, config['model']['filters']*2, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*4, config['model']['filters']*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']*2, config['model']['filters']*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']*2),
            nn.ReLU(inplace=True)
        )
        
        self.up1 = nn.ConvTranspose2d(config['model']['filters']*2, config['model']['filters'], kernel_size=4, stride=2, padding=1)
        self.conv1 = nn.Sequential(
            nn.Conv2d(config['model']['filters']*2, config['model']['filters'], kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters'], config['model']['filters'], kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']),
            nn.ReLU(inplace=True)
        )
        
        # output layer 
        self.output = nn.Sequential(
            nn.Conv2d(config['model']['filters'], config['model']['filters']//2, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']//2),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']//2, config['model']['filters']//4, kernel_size=3, padding=1),
            nn.BatchNorm2d(config['model']['filters']//4),
            nn.ReLU(inplace=True),
            nn.Conv2d(config['model']['filters']//4, 1, kernel_size=1),
            nn.Sigmoid()
        )
        
        # Initialize weights for better training
        self._init_weights()
    
    def _init_weights(self):#initilising the weight using kaiming normal for conv layers and for batch kept it 1 and 0 
        for w in self.modules(): #here w is the layers - as submodules inside the module - which contain all the layers 
            if isinstance(w, nn.Conv2d) or isinstance(w, nn.ConvTranspose2d):
                nn.init.kaiming_normal_(w.weight, mode='fan_out', nonlinearity='relu')
                if w.bias is not None:
                    nn.init.constant_(w.bias, 0)
            elif isinstance(w, nn.BatchNorm2d):
                nn.init.constant_(w.weight, 1)
                nn.init.constant_(w.bias, 0)
    
    def forward(self, x):
        #encoder 
        d1 = self.down1(x)
        p1 = self.pool1(d1)
        
        d2 = self.down2(p1)
        p2 = self.pool2(d2)
        
        d3 = self.down3(p2)
        p3 = self.pool3(d3)
        
        d4 = self.down4(p3)
        p4 = self.pool4(d4)
        
        #bottleneck
        b = self.bottleneck(p4)
        
        #decoder- added skip connections 
        u4 = self.up4(b)
        u4 = torch.cat([u4, d4], dim=1)
        c4 = self.conv4(u4)
        
        u3 = self.up3(c4)
        u3 = torch.cat([u3, d3], dim=1)
        c3 = self.conv3(u3)
        
        u2 = self.up2(c3)
        u2 = torch.cat([u2, d2], dim=1)
        c2 = self.conv2(u2)
        
        u1 = self.up1(c2)
        u1 = torch.cat([u1, d1], dim=1)
        c1 = self.conv1(u1)
        
        # Final output
        out = self.output(c1)
        
        return out

def find_dataset(base_dirs):
    for base_dir in base_dirs:
        if (os.path.exists(os.path.join(base_dir, 'train')) and 
            os.path.exists(os.path.join(base_dir, 'test'))):
            return base_dir
        
        if os.path.isdir(base_dir):
            for subdir in os.listdir(base_dir):
                subdir_path = os.path.join(base_dir, subdir)
                if not os.path.isdir(subdir_path):
                    continue
                    
                if (os.path.exists(os.path.join(subdir_path, 'train')) and 
                    os.path.exists(os.path.join(subdir_path, 'test'))):
                    return subdir_path
    return None

def better_enhance_depth(depth):
    """
    Enhanced depth map enhancement that preserves edges and reduces blocky artifacts
    """
    # Step 1: Normalize depth map
    depth_min = depth.min()
    depth_max = depth.max()
    if depth_max > depth_min:
        depth = (depth - depth_min) / (depth_max - depth_min)
    
    try:
        # Step 2: Apply bilateral filter to smooth while preserving edges
        # This is key to reducing the "boxy" artifacts
        depth_smooth = cv2.bilateralFilter(depth.astype(np.float32), d=7, sigmaColor=0.1, sigmaSpace=5)
        
        # Step 3: Convert to uint8 for processing
        depth_uint8 = (depth_smooth * 255).astype(np.uint8)
        
        # Step 4: Enhance local contrast with CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        depth_contrast = clahe.apply(depth_uint8)
        depth_contrast = depth_contrast.astype(np.float32) / 255.0
        
        # Step 5: Enhance edges to make objects more visible
        kernel_edge = np.array([[0, -1, 0], 
                                [-1, 5, -1], 
                                [0, -1, 0]], dtype=np.float32)
        depth_edge = cv2.filter2D(depth_contrast, -1, kernel_edge)
        
        # Step 6: Blend original and enhanced versions
        depth_blend = 0.7 * depth_edge + 0.3 * depth_smooth
        
        # Step 7: Final gentle smoothing to reduce any noise
        depth_final = cv2.GaussianBlur(depth_blend, (3, 3), 0.5)
        
    except Exception as e:
        print(f"Error in depth enhancement: {e}")
        depth_final = depth  # Fallback to original if enhancement fails
    
    return np.clip(depth_final, 0, 1)

def train_model(model, train_loader, val_loader, device, num_epochs=30, lr=0.0002, start_epoch=0):
    def improved_depth_loss(pred, target):
        # Ensure same size
        if pred.size() != target.size():
            target = F.interpolate(target, size=pred.size()[2:], mode='bilinear', align_corners=True)
        
        # Basic L1 loss
        l1_loss = F.l1_loss(pred, target)
        
        try:
            # Edge-aware loss components
            sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=device).view(1, 1, 3, 3)
            sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=device).view(1, 1, 3, 3)
            
            # Gradient loss to preserve edges
            pred_grad_x = F.conv2d(pred, sobel_x, padding=1)
            pred_grad_y = F.conv2d(pred, sobel_y, padding=1)
            target_grad_x = F.conv2d(target, sobel_x, padding=1)
            target_grad_y = F.conv2d(target, sobel_y, padding=1)
            
            grad_loss = F.l1_loss(pred_grad_x, target_grad_x) + F.l1_loss(pred_grad_y, target_grad_y)
            
            # Add simple smoothness loss to reduce "boxy" artifacts
            pred_dx = torch.abs(pred[:, :, :, :-1] - pred[:, :, :, 1:])
            pred_dy = torch.abs(pred[:, :, :-1, :] - pred[:, :, 1:, :])
            smooth_loss = torch.mean(pred_dx) + torch.mean(pred_dy)
            
            # Combine all losses with weights
            total_loss = 0.7 * l1_loss + 0.2 * grad_loss + 0.1 * smooth_loss
        except Exception as e:
            print(f"Loss calculation error: {e}, using basic L1 loss")
            total_loss = l1_loss
        
        return total_loss
    
    # Use AdamW instead of Adam for better weight decay handling
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # LR scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
    
    best_loss = float('inf')
    best_model_path = 'best_depth_model.pth'
    
    for epoch in range(start_epoch, num_epochs):
        epoch_start_time = time.time()
        
        # Training phase
        model.train()
        train_loss = 0
        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]',bar_format='{l_bar}{bar}|{n_fmt}/{total_fmt}[{postfix}]')
        
        for batch_idx, (rgb, depth) in enumerate(train_bar):
            try:
                rgb, depth = rgb.to(device), depth.to(device)
                
                # Forward pass
                optimizer.zero_grad()
                pred_depth = model(rgb)
                
                # Calculate loss
                loss = improved_depth_loss(pred_depth, depth)
                
                # Backward pass
                loss.backward()
                
                # Gradient clipping to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                # Update weights
                optimizer.step()
                
                # Update running loss
                train_loss += loss.item()
                train_bar.set_postfix({'loss': loss.item()})
                
            except Exception as e:
                print(f"Error in batch: {e}")
                continue
                
        # Calculate average training loss
        train_loss /= len(train_loader)
        
        # Clear memory- session timed out
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        # Validation phase
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for rgb, depth in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]'):
                rgb, depth = rgb.to(device), depth.to(device)
                pred_depth = model(rgb)
                loss = improved_depth_loss(pred_depth, depth)
                val_loss += loss.item()
        
        # Calculate average validation loss
        val_loss /= len(val_loader)
        
        # Update learning rate
        scheduler.step(val_loss)
        
        
        # Print epoch results
        print(f'\nEpoch {epoch+1}/{num_epochs}:')
        print(f'Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')
        
        # Save best model
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved with validation loss {val_loss:.4f}")

    print("Training completed!")

    if os.path.exists('/kaggle'):
        try:
            from IPython.display import FileLink
            display(FileLink(best_model_path))
            print(f"link to download your BEST model (val_loss: {best_loss:.4f})")
        except:
            pass

    return model, best_model_path
    
def main():
    set_seed(42)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using {device}")
    
    # Use our improved model
    model = ImprovedDepthModel().to(device)
    
    possible_paths = []
    if os.path.exists('/kaggle/input'):
        for dataset in os.listdir('/kaggle/input'):
            possible_paths.append(os.path.join('/kaggle/input', dataset))
        possible_paths.append('/kaggle/input')
    
    root_folder = find_dataset(possible_paths)
    if root_folder is None:
        print("Dataset not found. Please check the path.")
        return
        
    print(f"Found dataset at: {root_folder}")
    
    train_dataset = NYUDepthDataset(root_folder, split='train')
    val_dataset = NYUDepthDataset(root_folder, split='test')
    
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print("Empty dataset found")
        return
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config['dataset']['batch_size'],
        shuffle=True,
        num_workers=config['dataset']['workers'],
        drop_last=True,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['dataset']['batch_size'],
        shuffle=False,
        num_workers=config['dataset']['workers'],
        drop_last=False,
        pin_memory=True
    )
    
    print(f"Created data loaders - Train: {len(train_loader)} batches, Validation: {len(val_loader)} batches")
    
    start_epoch =0
    
    print("\n=== Starting Training ===")
    model, best_model_path = train_model(
        model,
        train_loader,
        val_loader,
        device,
        num_epochs=config['training']['epochs'],
        lr=config['training']['lr'],
        start_epoch=start_epoch
    )
    
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        print("Best model loaded")
    
    if os.path.exists('simple_depth_model.pth'):
        print("Model ready for use")
    
    print("\n=== Model Ready for Depth Estimation ===")

if __name__ == '__main__':
    main()
